# Full fine-tuning for statute identification


In [ ]:
# Keep the CUDA-enabled torch supplied by your SageMaker kernel; do not replace it with CPU torch.
import importlib.util, subprocess, sys
requirements = {'numpy': 'numpy>=1.26,<3', 'tokenizers': 'tokenizers>=0.20,<1',
                'huggingface_hub': 'huggingface_hub>=0.26,<2', 'pandas': 'pandas>=2,<4',
                'matplotlib': 'matplotlib>=3.8,<4'}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
assert importlib.util.find_spec('torch'), 'Select a Kaggle PyTorch kernel first.'


## 1. Configuration
Default: 384-token overlapping windows, at most 4 uniformly spaced windows per document,
mean token pooling and learned attention across windows. This covers beginning/middle/end,
but is **sampled context**, not the complete document. Set `max_chunks=None` for all windows;
monitor memory and run time. Coverage is recorded. Head-only and student-without-KD controls
are available as optional additional experiments. `smoke_only=True` is a pipeline check,
never a scored experiment. Budgeted training defaults to ONE epoch on a capped subset.


In [ ]:
from pathlib import Path
import os, sys, json, math, time, random, hashlib, tarfile, shutil, gc, copy, csv, zipfile
from contextlib import nullcontext
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from tokenizers import Tokenizer
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2,3), 'Use a Kaggle PyTorch 2.3+ kernel (torch.amp.GradScaler required).'

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/phase5_lsi_budgeted')
FOUNDATION_ARCHIVE = None  # Optional explicit Path to your .tar.gz input
FOUNDATION_DIR = None      # Or explicit Path containing extracted model.pt + tokenizer.json
LOCAL_DATA = Path('/kaggle/working/capstone_data/Phase_5_LSI')  # Already downloaded by your snippet
assert all((LOCAL_DATA / p).is_file() for p in ('Train/train.json','Train/dev.json','Test/test.json')), 'Existing working-directory Phase 5 files are missing; rerun your download snippet.'
RESUME_FROM = None         # Optional Path('/kaggle/input/<saved-output>/phase5_lsi_budgeted')

def discover_kaggle_inputs(root, archive=None, foundation_dir=None, local_data=None):
    root = Path(root)
    if archive is None and foundation_dir is None:
        archives = sorted(root.rglob('lexy-pytorch-default-v1.tar.gz'))
        # Ignore a saved output's foundation copy when looking for the original input.
        dirs = sorted(p.parent for p in root.rglob('model.pt')
                      if (p.parent/'tokenizer.json').is_file() and p.parent.name != 'foundation')
        if len(archives) == 1:
            archive = archives[0]
        elif not archives and len(dirs) == 1:
            foundation_dir = dirs[0]
        else:
            raise ValueError(f'Attach one Phase 1 input or set FOUNDATION_ARCHIVE/FOUNDATION_DIR. '
                             f'Archives: {archives}; extracted candidates: {dirs}')
    if local_data is None:
        roots = sorted({p.parent.parent for p in root.rglob('train.json')
                        if p.parent.name == 'Train' and (p.parent/'dev.json').is_file()
                        and (p.parent.parent/'Test'/'test.json').is_file()})
        if len(roots) > 1: raise ValueError(f'Multiple official datasets; set LOCAL_DATA: {roots}')
        local_data = roots[0] if roots else None
    return archive, foundation_dir, local_data

FOUNDATION_ARCHIVE, FOUNDATION_DIR, LOCAL_DATA = discover_kaggle_inputs(
    INPUT_ROOT, FOUNDATION_ARCHIVE, FOUNDATION_DIR, LOCAL_DATA)
if RESUME_FROM is not None:
    source = Path(RESUME_FROM).resolve()
    assert source.is_relative_to(INPUT_ROOT.resolve()) and source.is_dir()
    assert not WORK.exists() or not any(WORK.iterdir()), 'Resume copy requires empty WORK; do not overwrite an active run.'
    shutil.copytree(source, WORK, dirs_exist_ok=True)
if LOCAL_DATA is None and not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        raise RuntimeError('Attach the official Phase_5_LSI dataset, or grant this notebook access '
                           'to the HF_TOKEN Kaggle secret and enable Internet.') from None
os.environ.setdefault('HF_HOME', '/kaggle/temp/huggingface')
print('Model input:', FOUNDATION_ARCHIVE or FOUNDATION_DIR, '| Local data:', LOCAL_DATA)
HF_REVISION = 'main'  # resolved to an immutable commit and recorded at download time
S3_BACKUP = None  # Kaggle uses Save Version/downloads; leave disabled.
EXPERIMENTS = ["full"]
BUDGETS = {
    'full': dict(minutes=180, max_docs=15000),
    'partial': dict(minutes=90, max_docs=7000),
    'lorma': dict(minutes=90, max_docs=7000),
    'adapter': dict(minutes=90, max_docs=7000),
    'prompt': dict(minutes=45, max_docs=3500),
    'kd': dict(minutes=45, max_docs=3500),
    'head': dict(minutes=45, max_docs=3500),  # optional control
    'student': dict(minutes=45, max_docs=3500),  # optional no-KD control
}
EXPORT_ALL_VARIANTS = False  # False: predict the complete test set only with the selected winner
CFG = dict(seed=42, epochs=1, sequence_length=384, stride=256, max_chunks=4,
           accumulation=8, lr_full=2e-5, lr_peft=3e-4, weight_decay=0.01,
           warmup_fraction=0.06, rank=8, alpha=8, adapter_dim=64, prompt_tokens=16,
           partial_layers=3, student_fraction=0.5, temperature=2.0, kd_weight=0.5,
           pos_weight_cap=20.0, grad_checkpoint=True, pool='attention',
           resume=True, smoke_only=False, smoke_train=16, smoke_dev=12,
           selection_docs=256, calibration_docs=512, checkpoint_seconds=600)
WORK.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16
def amp():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if DEVICE.type == 'cuda' else nullcontext()
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 << 20), b''): h.update(block)
    return h.hexdigest()
def atomic_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8'); tmp.replace(path)
def atomic_save(obj, path):
    path = Path(path); tmp = path.with_suffix('.tmp'); torch.save(obj, tmp); tmp.replace(path)
def backup(directory):
    if not S3_BACKUP: return
    import boto3
    from urllib.parse import urlparse
    u = urlparse(S3_BACKUP)
    assert u.scheme == 's3' and u.netloc
    client = boto3.client('s3')
    for p in Path(directory).rglob('*'):
        if p.is_file() and not p.name.endswith('.tmp'):
            key = '/'.join([u.path.strip('/'), str(p.relative_to(WORK)).replace(os.sep, '/')]).lstrip('/')
            client.upload_file(str(p), u.netloc, key)
seed_all(CFG['seed'])
print('Device:', DEVICE, 'Torch:', torch.__version__, 'Output:', WORK)
if DEVICE.type != 'cuda': print('CPU mode: use for smoke testing, not the full training schedule.')


## 2. Recovered original Phase 1 architecture
Exact model-definition cell recovered from the supplied `phase-1.ipynb`. Checkpoint weights
are loaded strictly, so structural mismatches fail before training. Only load your trusted archive.


In [ ]:
"""LEXWeave: a project-specific experimental causal LM, not a pretrained model.

Parallel token attention, strictly completed-chunk memory, and routed dilated
convolution, combined by a token-dependent gate. See notebook for hypotheses,
prior work, limitations, and controlled ablations. No claim of literature novelty.
"""
import math
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        z = x.float()
        return (z * torch.rsqrt(z.square().mean(-1, keepdim=True) + 1e-6)).to(x.dtype) * self.weight


class Rotary(nn.Module):
    def __init__(self, head_dim):
        super().__init__()
        assert head_dim % 2 == 0
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer('inv', inv, persistent=False)

    def forward(self, x, positions):
        a = torch.outer(positions.float(), self.inv.float())
        c = a.cos().to(x.dtype)[None, None]
        s = a.sin().to(x.dtype)[None, None]
        u, v = x[..., ::2], x[..., 1::2]
        return torch.stack((u*c-v*s, u*s+v*c), -1).flatten(-2)


class Attention(nn.Module):
    """Exact causal token attention is retained for names, numbers and citations."""
    def __init__(self, dim, heads, kv_heads):
        super().__init__()
        assert dim % heads == 0 and heads % kv_heads == 0
        self.h, self.k, self.dh = heads, kv_heads, dim // heads
        self.q = nn.Linear(dim, dim, bias=False)
        self.kv = nn.Linear(dim, 2 * kv_heads * self.dh, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.rope = Rotary(self.dh)

    def forward(self, x):
        b, t, d = x.shape
        q = self.q(x).view(b, t, self.h, self.dh).transpose(1, 2)
        k, v = self.kv(x).view(b, t, 2, self.k, self.dh).unbind(2)
        k, v = k.transpose(1, 2), v.transpose(1, 2)
        pos = torch.arange(t, device=x.device)
        q, k = self.rope(q, pos), self.rope(k, pos)
        if self.h != self.k:
            k = k.repeat_interleave(self.h // self.k, dim=1)
            v = v.repeat_interleave(self.h // self.k, dim=1)
        z = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.)
        return self.out(z.transpose(1, 2).contiguous().view(b, t, d))


class ChunkMemory(nn.Module):
    """Learn a salience-weighted vector per chunk; read only COMPLETED past chunks.

    Memory is recomputed inside the current context, not carried between batches.
    A fixed zero slot makes the first chunk well-defined without all-masked rows.
    """
    def __init__(self, dim, mem_dim=128, heads=4, chunk=32, learned_pool=True):
        super().__init__()
        assert mem_dim % heads == 0
        self.dim, self.md, self.h = dim, mem_dim, heads
        self.dh, self.chunk = mem_dim // heads, chunk
        self.score = nn.Linear(dim, 1, bias=False) if learned_pool else None
        self.q = nn.Linear(dim, mem_dim, bias=False)
        self.kv = nn.Linear(dim, 2*mem_dim, bias=False)
        self.out = nn.Linear(mem_dim, dim, bias=False)
        self.rope = Rotary(self.dh)

    def pool(self, x):
        b, t, d = x.shape
        c = self.chunk
        n = (t+c-1)//c
        padded = F.pad(x, (0, 0, 0, n*c-t)).view(b, n, c, d)
        valid = torch.arange(n*c, device=x.device).view(1, n, c) < t
        if self.score is None:
            w = valid.float().expand(b, -1, -1)
            w = w / w.sum(-1, keepdim=True).clamp_min(1)
        else:
            logits = self.score(padded).squeeze(-1).float()
            logits = logits.masked_fill(~valid, -float('inf'))
            w = logits.softmax(-1)
        summaries = (padded * w.to(padded.dtype).unsqueeze(-1)).sum(-2)
        ends = (torch.arange(n, device=x.device)+1)*c - 1
        return summaries, ends

    def forward(self, x):
        b, t, _ = x.shape
        summary, ends = self.pool(x)
        n = summary.shape[1]
        q = self.q(x).view(b, t, self.h, self.dh).transpose(1, 2)
        k, v = self.kv(summary).view(b, n, 2, self.h, self.dh).unbind(2)
        k, v = k.transpose(1, 2), v.transpose(1, 2)
        pos = torch.arange(t, device=x.device)
        q, k = self.rope(q, pos), self.rope(k, ends)
        # Null slot: exactly zero, no learned information from any future token.
        k = torch.cat((k.new_zeros(b, self.h, 1, self.dh), k), dim=2)
        v = torch.cat((v.new_zeros(b, self.h, 1, self.dh), v), dim=2)
        allowed = ends[None, :] < pos[:, None]
        allowed = torch.cat((torch.ones(t, 1, dtype=torch.bool, device=x.device), allowed), -1)
        z = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed[None, None], dropout_p=0.)
        return self.out(z.transpose(1, 2).contiguous().view(b, t, self.md))


class PhraseMixer(nn.Module):
    """Token-gated, left-padded depthwise convolutions at multiple spacings."""
    def __init__(self, dim, inner=128, dilations=(1, 4, 16)):
        super().__init__()
        self.dilations = tuple(dilations)
        self.up = nn.Linear(dim, inner, bias=False)
        self.conv = nn.ModuleList([nn.Conv1d(inner, inner, 3, dilation=r,
                                            groups=inner, bias=False) for r in self.dilations])
        self.scale_gate = nn.Linear(inner, len(self.dilations), bias=False) if len(self.dilations)>1 else None
        self.out = nn.Linear(inner, dim, bias=False)

    def forward(self, x):
        z = F.silu(self.up(x))
        gate = self.scale_gate(z).float().softmax(-1).to(z.dtype) if self.scale_gate is not None else None
        y = torch.zeros_like(z)
        for i, (conv, r) in enumerate(zip(self.conv, self.dilations)):
            branch = conv(F.pad(z.transpose(1, 2), (2*r, 0))).transpose(1, 2)
            y = y + (branch * gate[..., i:i+1] if gate is not None else branch)
        return self.out(F.silu(y))


class WeaveBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, a = cfg['dim'], cfg['ablation']
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = Attention(d, cfg['heads'], cfg['kv_heads'])
        self.memory = None if a in ('no_memory', 'plain') else ChunkMemory(
            d, cfg['mem_dim'], cfg['mem_heads'], cfg['chunk'], learned_pool=a != 'mean_pool')
        self.phrase = None if a in ('no_conv', 'plain') else PhraseMixer(
            d, cfg['conv_dim'], (1,) if a == 'single_scale' else cfg['dilations'])
        self.active = [0] + ([1] if self.memory is not None else []) + ([2] if self.phrase is not None else [])
        self.router = nn.Linear(d, 3, bias=False) if a not in ('fixed_mix', 'plain') else None
        self.register_buffer('route_prior', torch.tensor([2., 0., 0.]), persistent=True)
        self.chunk = cfg['chunk']
        self.up = nn.Linear(d, 2*cfg['ff_dim'], bias=False)
        self.down = nn.Linear(cfg['ff_dim'], d, bias=False)
        self.record_routes = False
        self.last_routes = None

    def route_weights(self, x):
        b, t, _ = x.shape
        logits = self.route_prior[None, None].expand(b, t, 3)
        if self.router is not None: logits = logits + self.router(x).float()
        enabled = torch.zeros(t, 3, dtype=torch.bool, device=x.device)
        enabled[:, self.active] = True
        # Chunk memory carries no past summary during the first chunk.
        enabled[:, 1] &= torch.arange(t, device=x.device) >= self.chunk
        return logits.masked_fill(~enabled[None], -float('inf')).softmax(-1).to(x.dtype)

    def forward(self, x):
        z = self.n1(x)
        weights = self.route_weights(z)
        mixed = weights[..., 0:1] * self.attn(z)
        if self.memory is not None: mixed = mixed + weights[..., 1:2] * self.memory(z)
        if self.phrase is not None: mixed = mixed + weights[..., 2:3] * self.phrase(z)
        if self.record_routes and not self.training:
            self.last_routes = weights.detach().float().mean((0, 1)).cpu()
        x = x + mixed
        a, b = self.up(self.n2(x)).chunk(2, -1)
        return x + self.down(F.silu(a) * b)


class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg, self.kind = dict(cfg), 'lexweave'
        self.grad_ckpt = cfg.get('grad_checkpoint', True)
        self.emb = nn.Embedding(cfg['vocab'], cfg['dim'])
        self.blocks = nn.ModuleList([WeaveBlock(cfg) for _ in range(cfg['layers'])])
        self.norm = RMSNorm(cfg['dim'])
        self.apply(self.init_weights)
        # Mild residual initialization keeps the extra paths from overwhelming input.
        residual_std = 0.02 / math.sqrt(2*cfg['layers'])
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=residual_std)
            nn.init.normal_(block.down.weight, std=residual_std)
            if block.memory is not None:
                nn.init.normal_(block.memory.out.weight, std=residual_std)
                if block.memory.score is not None: nn.init.zeros_(block.memory.score.weight)
            if block.phrase is not None:
                nn.init.normal_(block.phrase.out.weight, std=residual_std)
                if block.phrase.scale_gate is not None: nn.init.zeros_(block.phrase.scale_gate.weight)
            if block.router is not None: nn.init.zeros_(block.router.weight)

    @staticmethod
    def init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding, nn.Conv1d)):
            nn.init.normal_(m.weight, std=0.02)
            if getattr(m, 'bias', None) is not None: nn.init.zeros_(m.bias)

    def hidden(self, ids, src=None, memory=None):
        if ids.shape[1] > self.cfg['context']: raise ValueError('Input exceeds configured context.')
        x = self.emb(ids)
        for block in self.blocks:
            x = checkpoint(block, x, use_reentrant=False) if self.training and self.grad_ckpt else block(x)
        return self.norm(x)

    def forward(self, ids, src=None, targets=None, memory=None):
        x = self.hidden(ids)
        if targets is None: return F.linear(x[:, -1], self.emb.weight).float()
        def ce(h, y):
            logits = F.linear(h, self.emb.weight).float()
            return F.cross_entropy(logits.reshape(-1, self.cfg['vocab']), y.reshape(-1), reduction='sum')
        total = x.new_zeros((), dtype=torch.float32)
        for j in range(0, x.shape[1], 128):
            h, y = x[:, j:j+128], targets[:, j:j+128]
            total = total + (checkpoint(ce, h, y, use_reentrant=False) if self.training and self.grad_ckpt else ce(h, y))
        return total / targets.ne(-100).sum().clamp_min(1)


In [ ]:
def extract_foundation(archive, destination):
    assert archive.is_file(), f'Upload your Phase 1 archive: {archive}'
    destination.mkdir(parents=True, exist_ok=True)
    # Read only two known files; never execute code or extract arbitrary archive paths.
    with tarfile.open(archive, 'r:gz') as tf:
        for name in ('model.pt', 'tokenizer.json'):
            hits = [m for m in tf.getmembers() if Path(m.name).name == name and m.isfile()]
            if len(hits) != 1: raise ValueError(f'Expected exactly one {name}, found {len(hits)}')
            with tf.extractfile(hits[0]) as src, open(destination / name, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    return destination

if FOUNDATION_DIR is not None:
    FOUNDATION = Path(FOUNDATION_DIR)  # read-only input is fine; never execute its Python files
    assert all((FOUNDATION / n).is_file() for n in ('model.pt', 'tokenizer.json'))
else:
    FOUNDATION = extract_foundation(Path(FOUNDATION_ARCHIVE), WORK / 'foundation')
# weights_only=False is necessary for this trusted Phase 1 checkpoint's metadata.
BASE = torch.load(FOUNDATION / 'model.pt', map_location='cpu', weights_only=False)
assert {'model_config', 'model'} <= BASE.keys(), f'Unexpected checkpoint keys: {BASE.keys()}'
BASE_CFG = dict(BASE['model_config'])
tokenizer = Tokenizer.from_file(str(FOUNDATION / 'tokenizer.json'))
tokenizer.no_truncation(); tokenizer.no_padding()
PAD, BOS, EOS = [tokenizer.token_to_id(t) for t in ('<pad>', '<bos>', '<eos>')]
assert None not in (PAD, BOS, EOS)
assert tokenizer.get_vocab_size() == BASE_CFG['vocab']
assert CFG['sequence_length'] + CFG['prompt_tokens'] <= BASE_CFG['context']
BASE_SHA = sha(FOUNDATION / 'model.pt')
TOK_SHA = sha(FOUNDATION / 'tokenizer.json')
probe = LegalLM(BASE_CFG); probe.load_state_dict(BASE['model'], strict=True)
print('Strict Phase 1 load passed:', f'{sum(p.numel() for p in probe.parameters()):,}', 'parameters')
del probe; gc.collect()


## 3. Official data, exact IDs, and label vocabulary
`train.json` and `dev.json` are under `Phase_5_LSI/Train`; `test.json` is under `Test`.
The public file listing was verified, but downloading contents requires dataset permission.
Authorize access on Hugging Face, then add and enable the `HF_TOKEN` Kaggle secret.
We accept a JSON list of records or an ID-keyed mapping, reject duplicate IDs and malformed
labels, and never silently discard samples. Integer statute IDs are preserved, not assumed contiguous.
Repeated statute IDs within a record are deduplicated in first-occurrence order and counted.
Vocabulary is the union of official train/dev labels. Dev-only statutes are reported; they have
no positive training signal. The official evaluator derives its label dimension from reference
labels, so local fixed-vocabulary metrics can differ from the hidden leaderboard metric.


In [ ]:
def resolve_data():
    if LOCAL_DATA is not None:
        root = Path(LOCAL_DATA)
        return root, {'source': 'local', 'root': str(root)}
    from huggingface_hub import HfApi, snapshot_download
    token = os.environ.get('HF_TOKEN')
    if not token: raise RuntimeError('Set the HF_TOKEN Kaggle secret or attach local data.')
    repo = 'Exploration-Lab/CS-781-Capstone'
    revision = HfApi().dataset_info(repo, revision=HF_REVISION, token=token).sha
    root = Path(snapshot_download(repo, repo_type='dataset', revision=revision, token=token,
                                 allow_patterns=['Phase_5_LSI/**'])) / 'Phase_5_LSI'
    return root, {'source': repo, 'revision': revision}

def read_records(path, labelled):
    raw = json.loads(Path(path).read_text(encoding='utf-8'))
    if isinstance(raw, dict):
        converted = []
        for k, v in raw.items():
            if not isinstance(v, dict): raise ValueError('ID-keyed entries must be objects')
            if 'id' in v and v['id'] != k: raise ValueError('Mapping key/id mismatch')
            converted.append(dict(v, id=k))
        raw = converted
    if not isinstance(raw, list) or not raw: raise ValueError(f'Empty/unsupported data: {path}')
    seen = set()
    duplicate_label_records = duplicate_label_entries = 0
    for r in raw:
        if not isinstance(r, dict) or not isinstance(r.get('id'), str):
            raise ValueError('Expected a string id; never coerce official IDs')
        if r['id'] in seen: raise ValueError(f'Duplicate ID: {r["id"]}')
        seen.add(r['id'])
        if not isinstance(r.get('text'), str): raise ValueError('Expected text string')
        if labelled:
            y = r.get('labels')
            if not isinstance(y, list) or not y or any(type(v) is not int or v < 0 for v in y):
                raise ValueError(f'Expected nonempty nonnegative integer labels: {r["id"]}')
            unique_labels = list(dict.fromkeys(y))
            if len(unique_labels) != len(y):
                duplicate_label_records += 1
                duplicate_label_entries += len(y) - len(unique_labels)
            r['labels'] = unique_labels
        elif 'labels' in r:
            raise ValueError('Test contains labels: confirm that this is the official unlabeled test file')
    if duplicate_label_entries:
        print(f'{Path(path).name}: removed {duplicate_label_entries} repeated statute IDs '
              f'across {duplicate_label_records} records; all records retained.')
    return raw

DATA_ROOT, DATA_PROVENANCE = resolve_data()
DATA_FILES = {s: DATA_ROOT / p for s, p in {'train':'Train/train.json', 'dev':'Train/dev.json', 'test':'Test/test.json'}.items()}
records = {s: read_records(p, s != 'test') for s, p in DATA_FILES.items()}
for a, b in [('train','dev'), ('train','test'), ('dev','test')]:
    overlap = {r['id'] for r in records[a]} & {r['id'] for r in records[b]}
    assert not overlap, f'{a}/{b} ID overlap: {list(overlap)[:5]}'
LABELS = sorted({v for s in ('train','dev') for r in records[s] for v in r['labels']})
L2I = {v:i for i,v in enumerate(LABELS)}
def targets(rows):
    y = np.zeros((len(rows), len(LABELS)), dtype=np.float32)
    for i, r in enumerate(rows):
        for label in r['labels']: y[i, L2I[label]] = 1
    return y
Y_TRAIN, Y_DEV = targets(records['train']), targets(records['dev'])
support = Y_TRAIN.sum(0)
DATA_PROVENANCE['sha256'] = {s:sha(p) for s,p in DATA_FILES.items()}
DATA_PROVENANCE['counts'] = {s:len(r) for s,r in records.items()}
DATA_PROVENANCE['dev_only_labels'] = [LABELS[i] for i in np.flatnonzero(support == 0)]
# Detect exact duplicate text across splits before any training; record counts, not private text.
text_hashes = {s:{hashlib.sha256(r['text'].encode()).hexdigest() for r in rows} for s,rows in records.items()}
DATA_PROVENANCE['text_overlap'] = {f'{a}/{b}':len(text_hashes[a] & text_hashes[b]) for a,b in [('train','dev'),('train','test'),('dev','test')]}
atomic_json(DATA_PROVENANCE, WORK/'data_manifest.json')
atomic_json(LABELS, WORK/'labels.json')
print(DATA_PROVENANCE)
print('Statutes:', len(LABELS), 'Label count mean:', float(Y_TRAIN.sum(1).mean()))


## 4. Token cache and long-document representation
Windows are deterministic and shared by every experiment, including the teacher and student.
Short windows are run without padding, avoiding contamination from causal chunk-memory pooling.
Every document contributes one BCE loss regardless of its number of windows. Chunk count is capped
only when configured; the last window is always retained by uniform selection.


In [ ]:
def window_starts(n, payload, stride, max_chunks):
    assert payload > 0 and 0 < stride <= payload
    starts = list(range(0, max(n - payload, 0) + 1, stride))
    if starts[-1] + payload < n: starts.append(n - payload)
    if max_chunks is not None and len(starts) > max_chunks:
        assert max_chunks >= 2
        starts = [starts[i] for i in np.linspace(0, len(starts)-1, max_chunks).round().astype(int)]
    return starts

def make_windows(tokens):
    payload = CFG['sequence_length'] - 2
    starts = window_starts(len(tokens), payload, CFG['stride'], CFG['max_chunks'])
    return [[BOS] + tokens[s:s+payload] + [EOS] for s in starts]

def prepare_tokens(rows, split):
    key = hashlib.sha256((TOK_SHA + DATA_PROVENANCE['sha256'][split]).encode()).hexdigest()[:20]
    path = WORK / f'tokens_{split}_{key}.pt'
    if path.exists():
        data = torch.load(path, map_location='cpu', weights_only=False)
    else:
        data = []
        for i in range(0, len(rows), 32):
            data.extend(e.ids for e in tokenizer.encode_batch([r['text'] for r in rows[i:i+32]], add_special_tokens=False))
            if i % 1024 == 0: print(f'Tokenizing {split}: {min(i+32,len(rows))}/{len(rows)}',flush=True)
        atomic_save(data, path)
    assert len(data) == len(rows)
    return data

TOKENS = {s:prepare_tokens(rows,s) for s,rows in records.items()}
coverage = {}
for s, token_rows in TOKENS.items():
    stats = []
    for ts in token_rows:
        starts = window_starts(len(ts), CFG['sequence_length']-2, CFG['stride'], CFG['max_chunks'])
        end = covered = 0
        for start in starts:
            stop = min(start + CFG['sequence_length']-2, len(ts))
            covered += max(0, stop-max(start,end)); end = max(end,stop)
        stats.append([len(ts), len(starts), covered/max(1,len(ts))])
    a = np.array(stats)
    coverage[s] = dict(mean_tokens=float(a[:,0].mean()), max_tokens=int(a[:,0].max()),
                       mean_chunks=float(a[:,1].mean()), mean_token_coverage=float(a[:,2].mean()))
atomic_json(coverage, WORK/'coverage.json'); print(coverage)
# Split dev once: selection chooses epochs; calibration chooses thresholds, never training weights.
assert len(records['dev']) >= 4
order = np.random.default_rng(CFG['seed']).permutation(len(records['dev']))
SELECT_IDX, CAL_IDX = np.array_split(order, 2)
SELECT_IDX = SELECT_IDX[:CFG['selection_docs']]
CAL_IDX = CAL_IDX[:CFG['calibration_docs']]
TRAIN_IDX = np.arange(len(records['train']))
if CFG['smoke_only']:
    TRAIN_IDX = TRAIN_IDX[:CFG['smoke_train']]
    SELECT_IDX = SELECT_IDX[:max(2, CFG['smoke_dev']//2)]
    CAL_IDX = CAL_IDX[:max(2, CFG['smoke_dev']//2)]
atomic_json({'selection_ids':[records['dev'][i]['id'] for i in SELECT_IDX],
             'calibration_ids':[records['dev'][i]['id'] for i in CAL_IDX]}, WORK/'dev_partition.json')

# Build a shared nested order: early examples cover rare labels where possible, followed
# by a seeded random sample. Smaller budgets are prefixes of the same order, never file heads.
def training_order(y, indices, seed):
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(indices).tolist()
    counts = y[indices].sum(0)
    selected, used, covered = [], set(), set()
    for label in np.argsort(counts):
        if counts[label] == 0 or int(label) in covered: continue
        candidate = next(i for i in shuffled if y[i,label] > 0)
        if candidate not in used:
            selected.append(candidate); used.add(candidate)
            covered.update(np.flatnonzero(y[candidate]).tolist())
    return np.array(selected + [i for i in shuffled if i not in used], dtype=int)
TRAIN_ORDER = training_order(Y_TRAIN, TRAIN_IDX, CFG['seed'])
def method_indices(method):
    return TRAIN_ORDER[:BUDGETS[method]['max_docs']]
atomic_json({m:{**BUDGETS[m], 'selected_docs':len(method_indices(m)),
               'covered_labels':int((Y_TRAIN[method_indices(m)].sum(0)>0).sum())}
             for m in EXPERIMENTS}, WORK/'budget_plan.json')


## 5. Five distinct adaptations and a compact student
**LoRMA+** uses `(I + alpha/r * B @ A) @ W`, with B initially zero. This is a
multiplicative transformation of the frozen projection, matching additive rank inflation in
the linked paper. Bias is added after transformation. Applied to Linear layers in LM blocks.
**Adapters** are zero-initialized residual bottlenecks after each whole block (not claimed to
be the exact Houlsby placement). **Prompt tuning** prepends trainable embeddings before the
causal backbone; pooling excludes prompt positions. Backbone weights stay frozen.
**Partial** unfreezes the last configured blocks and final norm. **Full** trains all parameters.
Every method trains the same fresh multilabel head and chunk attention.
**KD** initializes a student by evenly selecting half the Phase 1 blocks, then trains with
supervised BCE plus temperature-scaled Bernoulli KL from the best non-KD teacher's train logits.
No dev or test teacher predictions enter the KD training loss. A no-KD pruned student control
can be added with `EXPERIMENTS += ['student']` to isolate the effect of distillation.


In [ ]:
class LoRMAPlus(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__(); self.base = base; self.scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        for p in self.base.parameters(): p.requires_grad = False
    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + self.scale * F.linear(F.linear(z, self.A), self.B)
        return z if self.base.bias is None else z + self.base.bias

def insert_lorma(module):
    count = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LoRMAPlus(child, CFG['rank'], CFG['alpha'])); count += 1
        else: count += insert_lorma(child)
    return count

class AdapterBlock(nn.Module):
    def __init__(self, base, dim, bottleneck):
        super().__init__(); self.base = base
        self.down = nn.Linear(dim, bottleneck); self.up = nn.Linear(bottleneck, dim)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        x = self.base(x)
        return x + self.up(F.gelu(self.down(x)))

class LSIModel(nn.Module):
    def __init__(self, method):
        super().__init__(); self.method = method
        self.lm = LegalLM(BASE_CFG)
        self.lm.load_state_dict(BASE['model'], strict=True)
        self.lm.grad_ckpt = CFG['grad_checkpoint']
        self.kept_layers = list(range(len(self.lm.blocks)))
        if method in ('kd','student'):
            n = max(1, int(len(self.lm.blocks) * CFG['student_fraction']))
            assert n < len(self.lm.blocks), 'Student must be smaller than Phase 1 model'
            self.kept_layers = np.linspace(0,len(self.lm.blocks)-1,n).round().astype(int).tolist()
            self.lm.blocks = nn.ModuleList([self.lm.blocks[i] for i in self.kept_layers])
        for p in self.lm.parameters(): p.requires_grad = method in ('full','kd','student')
        self.prompt = None
        if method == 'partial':
            for block in self.lm.blocks[-CFG['partial_layers']:]:
                for p in block.parameters(): p.requires_grad = True
            for p in self.lm.norm.parameters(): p.requires_grad = True
        elif method == 'lorma':
            assert insert_lorma(self.lm.blocks) > 0
        elif method == 'adapter':
            self.lm.blocks = nn.ModuleList([AdapterBlock(b, BASE_CFG['dim'], CFG['adapter_dim']) for b in self.lm.blocks])
        elif method == 'prompt':
            self.prompt = nn.Parameter(torch.randn(CFG['prompt_tokens'], BASE_CFG['dim']) * .02)
        elif method not in ('full','kd','student','head'):
            raise ValueError(method)
        d = BASE_CFG['dim']
        self.chunk_score = nn.Linear(d, 1)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(.1), nn.Linear(d,len(LABELS)))
    def encode(self, ids):
        x = self.lm.emb(ids)
        p = 0
        if self.prompt is not None:
            p = self.prompt.shape[0]
            x = torch.cat([self.prompt.unsqueeze(0).expand(x.shape[0],-1,-1),x], dim=1)
        assert x.shape[1] <= BASE_CFG['context']
        for block in self.lm.blocks:
            x = torch.utils.checkpoint.checkpoint(block,x,use_reentrant=False) if self.training and self.lm.grad_ckpt else block(x)
        return self.lm.norm(x)[:,p:].mean(1)
    def forward(self, windows):
        # Process windows separately: bounded per-window activation size, no padding assumptions.
        h = torch.cat([self.encode(torch.tensor(w, device=DEVICE).unsqueeze(0)) for w in windows], dim=0)
        if CFG['pool'] == 'attention':
            weights = self.chunk_score(h).float().softmax(0).to(h.dtype)
            doc = (h*weights).sum(0)
        elif CFG['pool'] == 'mean': doc = h.mean(0)
        elif CFG['pool'] == 'max': doc = h.max(0).values
        else: raise ValueError(CFG['pool'])
        return self.head(doc).float()

def new_model(method):
    seed_all(CFG['seed'])
    return LSIModel(method).to(DEVICE)


## 6. Metrics and threshold calibration
Report macro-F1 on the complete known label vocabulary, micro-F1 and exact-set accuracy.
Also report reference-active macro-F1 (only statutes present in that dev partition), because
absent labels and the evaluator's label dimension can change macro averaging. This is a local
implementation; compare with the supplied official evaluator before leaderboard submission.
Calibration tries a global threshold, then per-label thresholds only with ≥5 positive and
≥5 negative calibration examples. Rare labels fall back to the global threshold. No forced
top-1 or fixed label count. Empty predicted sets are allowed. Calibration scores are tuned,
not unbiased holdout estimates; selection scores at 0.5 are shown separately.


In [ ]:
def scores(y, pred):
    y, pred = y.astype(bool), pred.astype(bool)
    tp = (y & pred).sum(0); fp = (~y & pred).sum(0); fn = (y & ~pred).sum(0)
    f = np.divide(2*tp, 2*tp+fp+fn, out=np.zeros(len(tp), dtype=float), where=(2*tp+fp+fn)>0)
    active = y.sum(0)>0
    return {'macro_f1':float(f.mean()), 'micro_f1':float(2*tp.sum()/max(1,2*tp.sum()+fp.sum()+fn.sum())),
            'subset_accuracy':float((y==pred).all(1).mean()),
            'reference_active_macro_f1':float(f[active].mean()) if active.any() else 0.0}

def sigmoid(logits):
    return 1 / (1 + np.exp(-np.clip(logits, -50, 50)))

def calibrate(y, probabilities):
    grid = np.linspace(.05,.95,37)
    global_t = max(grid, key=lambda t:(scores(y, probabilities>=t)['macro_f1'], -abs(t-.5)))
    thresholds = np.full(y.shape[1], global_t, dtype=np.float32)
    for j in range(y.shape[1]):
        if y[:,j].sum() >= 5 and (1-y[:,j]).sum() >= 5:
            thresholds[j] = max(grid, key=lambda t:(scores(y[:,j:j+1], probabilities[:,j:j+1]>=t)['macro_f1'], -abs(t-global_t)))
    return float(global_t), thresholds

@torch.no_grad()
def predict_logits(model, split, indices):
    model.eval(); result=[]
    for i in indices:
        with amp(): z = model(make_windows(TOKENS[split][int(i)]))
        result.append(z.cpu().numpy())
    return np.stack(result)


## 7. Training, checkpoint recovery, and experiment evidence
Accumulation divides by the actual final group size. Time is checked at optimizer-group
boundaries; at most one group can run past the budget. Stops also honor the document/epoch cap.
Atomic recovery checkpoints every 10 minutes store the NEXT document cursor, optimizer,
scheduler, scaler and RNG. Resume replays at most the unsaved interval, not the whole epoch.
Budget consumption resumes from the last checkpoint. Interrupted unsaved work cannot be recovered.
One selection pass evaluates the final trained checkpoint; best.pt names that final candidate.
Calibration and teacher/test inference are outside the training timer. S3 backup occurs afterward.
New settings/data/budget require a new WORK directory; old epoch-only checkpoints are incompatible.


In [ ]:
def run_identity(method, teacher=None):
    cfg = {k:v for k,v in CFG.items() if k != 'resume'}
    return dict(method=method, cfg=cfg, base_sha256=BASE_SHA, tokenizer_sha256=TOK_SHA,
                data=DATA_PROVENANCE, labels=LABELS, teacher=teacher,
                budget=BUDGETS[method], implementation='phase5-lsi-budget-v2')

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else None)
def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'])
    if DEVICE.type=='cuda' and state['cuda'] is not None: torch.cuda.set_rng_state_all(state['cuda'])

def train_one(method, teacher_logits=None, teacher_info=None):
    folder = WORK / ('smoke' if CFG['smoke_only'] else 'runs') / method
    folder.mkdir(parents=True, exist_ok=True)
    ident = run_identity(method, teacher_info)
    manifest = folder/'experiment.json'
    if manifest.exists() and json.loads(manifest.read_text()) != ident:
        raise ValueError(f'Run configuration changed: choose a new WORK directory ({folder})')
    atomic_json(ident, manifest)
    indices = method_indices(method)
    atomic_json([records['train'][int(i)]['id'] for i in indices], folder/'training_ids.json')
    model = new_model(method)
    trainable = [p for p in model.parameters() if p.requires_grad]
    n_all, n_train = sum(p.numel() for p in model.parameters()), sum(p.numel() for p in trainable)
    lr = CFG['lr_full'] if method in ('full','partial','kd','student') else CFG['lr_peft']
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=CFG['weight_decay'])
    epochs = 1 if CFG['smoke_only'] else CFG['epochs']
    steps = epochs * math.ceil(len(indices)/CFG['accumulation'])
    warm = max(1, round(steps*CFG['warmup_fraction']))
    def factor(step):
        if step < warm: return (step+1)/warm
        return .1 + .9*.5*(1+math.cos(math.pi*min(1,(step-warm)/max(1,steps-warm))))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, factor)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type=='cuda' and AMP_DTYPE==torch.float16)
    counts = Y_TRAIN[indices].sum(0)
    pw = np.clip((len(indices)-counts)/np.maximum(counts,1),1,CFG['pos_weight_cap'])
    pos_weight = torch.tensor(pw, device=DEVICE)
    epoch=cursor=processed=0; elapsed=loss_sum=0.; done=False; stop_reason=None
    if CFG['resume'] and (folder/'last.pt').exists():
        state = torch.load(folder/'last.pt',map_location='cpu',weights_only=False)
        model.load_state_dict(state['model']); opt.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler'])
        epoch=state['epoch']; cursor=state['cursor']; elapsed=state['elapsed_seconds']
        processed=state['processed']; loss_sum=state['loss_sum']; done=state['done']
        stop_reason=state['stop_reason']; restore_rng(state['rng']); del state
        print(method, 'resume:', epoch+1, 'next document:', cursor, 'done:', done, flush=True)
    if method=='kd':
        assert teacher_logits is not None and teacher_logits.shape == (len(indices),len(LABELS))
        teacher_by_index = {int(i):teacher_logits[j] for j,i in enumerate(indices)}
    budget_seconds = BUDGETS[method]['minutes'] * 60
    assert budget_seconds > 0 and len(indices) > 0
    if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
    session_start=time.perf_counter(); previous_elapsed=elapsed
    last_saved=elapsed
    def spent():
        return previous_elapsed + time.perf_counter()-session_start
    def save_progress():
        # Cursor is always the NEXT accumulation group; gradients are already stepped/cleared.
        atomic_save({'model':model.state_dict(),'optimizer':opt.state_dict(),'scheduler':scheduler.state_dict(),
                     'scaler':scaler.state_dict(),'epoch':epoch,'cursor':cursor,
                     'elapsed_seconds':elapsed,'processed':processed,'loss_sum':loss_sum,
                     'done':done,'stop_reason':stop_reason,'rng':rng_state()}, folder/'last.pt')
    while not done and epoch < epochs:
        # Prefix order preserves the rare-label coverage stage even if the timer stops early.
        order = indices if epoch == 0 else np.random.default_rng(CFG['seed']+epoch).permutation(indices)
        model.train()
        while cursor < len(order):
            elapsed=spent()
            if elapsed >= budget_seconds:
                stop_reason='time_budget'; done=True; break
            group=order[cursor:cursor+CFG['accumulation']]
            opt.zero_grad(set_to_none=True)
            for i in group:
                y=torch.tensor(Y_TRAIN[int(i)],device=DEVICE)
                with amp():
                    z=model(make_windows(TOKENS['train'][int(i)]))
                    hard=F.binary_cross_entropy_with_logits(z,y,pos_weight=pos_weight)
                    loss=hard
                    if method=='kd':
                        temp=CFG['temperature']
                        q=torch.sigmoid(torch.tensor(teacher_by_index[int(i)],device=DEVICE)/temp).clamp(1e-6,1-1e-6)
                        soft=(F.binary_cross_entropy_with_logits(z/temp,q,reduction='none')
                              + q*q.log() + (1-q)*(1-q).log()).mean()*temp**2
                        loss=(1-CFG['kd_weight'])*hard+CFG['kd_weight']*soft
                if not torch.isfinite(loss): raise FloatingPointError(f'Nonfinite loss: {method}/{epoch}/{i}')
                loss_sum+=loss.item(); scaler.scale(loss/len(group)).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainable,1.)
            old_scale=scaler.get_scale(); scaler.step(opt); scaler.update()
            if scaler.get_scale() >= old_scale: scheduler.step()
            if DEVICE.type=='cuda': torch.cuda.synchronize()
            cursor+=len(group); processed+=len(group); elapsed=spent()
            if (cursor-len(group)) % (CFG['accumulation']*25) == 0:
                print(f'{method}: {cursor}/{len(order)} docs, {elapsed/60:.1f}/{budget_seconds/60:.0f} min',flush=True)
            if elapsed-last_saved >= CFG['checkpoint_seconds']:
                save_progress(); last_saved=elapsed
        if cursor >= len(order):
            epoch+=1; cursor=0
            if epoch >= epochs: done=True; stop_reason='document_epoch_cap'
        elapsed=spent(); save_progress()
    # Resume a finished run without adding evaluation/reload time to its training budget.
    if not done:
        done=True; stop_reason='document_epoch_cap'; elapsed=spent(); save_progress()
    print(method, 'stopped:',stop_reason, '| documents processed:',processed,
          '| training minutes:',round(elapsed/60,2),flush=True)
    # One post-training selection pass per variant. No hidden extra full-dev epoch evaluations.
    val_logits=predict_logits(model,'dev',SELECT_IDX)
    metrics=scores(Y_DEV[SELECT_IDX],sigmoid(val_logits)>=.5)
    row=dict(epoch=epoch,cursor=cursor,loss=loss_sum/max(1,processed),elapsed_seconds=elapsed,
             processed_documents=processed,stop_reason=stop_reason,**metrics)
    atomic_json([row],folder/'history.json')
    # Preserve stable best.pt bytes on reruns so the KD cache/identity remains valid.
    if not (folder/'best.pt').exists():
        atomic_save({'model':model.state_dict(),'method':method,'epoch':epoch,'selection_metrics':metrics,
                     'kept_layers':model.kept_layers,'identity':ident},folder/'best.pt')
    logits=predict_logits(model,'dev',CAL_IDX); probs=sigmoid(logits)
    gt,thresholds=calibrate(Y_DEV[CAL_IDX],probs)
    atomic_json({'label_ids':LABELS,'global_threshold':gt,'thresholds':thresholds.tolist()},folder/'thresholds.json')
    np.savez_compressed(folder/'calibration_predictions.npz',logits=logits,targets=Y_DEV[CAL_IDX],indices=CAL_IDX)
    summary=dict(method=method,total_parameters=n_all,trainable_parameters=n_train,
                 trainable_fraction=n_train/n_all,training_seconds=elapsed,
                 processed_documents=processed,selected_documents=len(indices),stop_reason=stop_reason,
                 budget_minutes=BUDGETS[method]['minutes'],
                 peak_gpu_gb=torch.cuda.max_memory_allocated()/2**30 if DEVICE.type=='cuda' else None,
                 best_selection_macro_f1=metrics['macro_f1'],calibration_global=scores(Y_DEV[CAL_IDX],probs>=gt),
                 calibration_per_label=scores(Y_DEV[CAL_IDX],probs>=thresholds))
    atomic_json(summary,folder/'summary.json'); backup(folder)
    del model,opt,trainable; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return folder,summary


def load_trained(folder):
    state=torch.load(Path(folder)/'best.pt',map_location='cpu',weights_only=False)
    if state['identity'] != run_identity(state['method'], state['identity']['teacher']):
        raise ValueError('Current notebook config/data differs from checkpoint. Restore its experiment settings.')
    model=new_model(state['method']); model.load_state_dict(state['model']); model.eval()
    return model


## 8. Correctness checks before expensive training
Tiny tensors verify exact LoRMA identity and formula, chunk coverage, and multilabel scoring.
A real-model one-document forward/backward checks the selected tokenizer and checkpoint together.


In [ ]:
def unit_checks():
    base=nn.Linear(5,7); layer=LoRMAPlus(copy.deepcopy(base),3,6)
    x=torch.randn(2,4,5)
    assert torch.allclose(layer(x),base(x),atol=1e-6)
    with torch.no_grad(): layer.B.normal_(std=.02)
    weight=(torch.eye(7)+2*layer.B@layer.A)@base.weight
    assert torch.allclose(layer(x),F.linear(x,weight,base.bias),atol=1e-5)
    layer(x).sum().backward(); assert layer.B.grad is not None and layer.base.weight.grad is None
    assert window_starts(1,10,8,None)==[0]
    assert window_starts(31,10,8,None)[-1]==21
    assert window_starts(100,10,8,2)==[0,90]
    y=np.array([[1,0],[0,1]])
    assert scores(y,y)['macro_f1']==1 and scores(y,np.zeros_like(y))['macro_f1']==0
    print('Formula, window and metric checks passed.')
unit_checks()
probe=new_model('lorma')
with amp(): probe_loss=probe(make_windows(TOKENS['train'][0])[:1]).square().mean()
probe_loss.backward()
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in probe.parameters() if p.requires_grad)
print('Real checkpoint forward/backward passed; start training below.')
del probe,probe_loss; gc.collect()
if DEVICE.type=='cuda': torch.cuda.empty_cache()


## 9. Execute experiments
Runs sequentially on one GPU. The KD teacher is the strongest completed non-student experiment
by selection-half macro-F1, not by the threshold-fitted calibration score. All train logits are
cached only for the capped KD subset, so teacher inference does not scan all 42,750 records. Cache identity
includes teacher weights and token-window settings. Completed epochs are reused on rerun.


In [ ]:
RESULTS={}
for method in EXPERIMENTS:
    if method not in ('kd',):
        RESULTS[method]=train_one(method)
if 'kd' in EXPERIMENTS:
    candidates={k:v for k,v in RESULTS.items() if k not in ('kd','student')}
    if not candidates: raise ValueError('Run at least one teacher method alongside kd (completed runs resume).')
    teacher_name=max(candidates,key=lambda k:candidates[k][1]['best_selection_macro_f1'])
    teacher_folder=candidates[teacher_name][0]
    teacher_info={'method':teacher_name,'checkpoint_sha256':sha(teacher_folder/'best.pt')}
    cache_key=hashlib.sha256(json.dumps(run_identity('kd',teacher_info),sort_keys=True).encode()).hexdigest()[:20]
    cache=WORK/f'teacher_train_{cache_key}.npy'
    if not cache.exists():
        teacher=load_trained(teacher_folder)
        teacher_logits=predict_logits(teacher,'train',method_indices('kd'))
        np.save(cache,teacher_logits)
        del teacher; gc.collect()
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
    teacher_logits=np.load(cache)
    RESULTS['kd']=train_one('kd',teacher_logits,teacher_info)
    assert RESULTS['kd'][1]['total_parameters'] < candidates[teacher_name][1]['total_parameters']
print('Completed:',list(RESULTS))


## 10. Per-label errors, efficiency, and comparison tables
The calibration-half results below are explicitly threshold-tuned. They are not hidden-test
scores. Compare rare/common statutes, label cardinality, frequent missed/extra labels and
co-occurrence. Inspect errors locally without adding external data. Add a `student` control,
or rerun into a new WORK with mean pooling/all windows to support causal claims about KD and
long-document representation. The six default runs alone confound method and tuned learning rate.


In [ ]:
import pandas as pd
from IPython.display import display, FileLink
comparison=[]
for method,(folder,summary) in RESULTS.items():
    payload=np.load(folder/'calibration_predictions.npz')
    p=sigmoid(payload['logits']); y=payload['targets'].astype(bool)
    payload.close()
    thresholds=np.array(json.loads((folder/'thresholds.json').read_text())['thresholds'])
    pred=p>=thresholds
    tp=(y&pred).sum(0); fp=(~y&pred).sum(0); fn=(y&~pred).sum(0)
    precision=tp/np.maximum(1,tp+fp); recall=tp/np.maximum(1,tp+fn)
    per_label=pd.DataFrame({'statute_id':LABELS,
                            'train_support':Y_TRAIN[method_indices(method)[:summary['processed_documents']]].sum(0).astype(int),
                            'official_train_support':support.astype(int),
                            'calibration_support':y.sum(0),'precision':precision,'recall':recall,
                            'f1':2*tp/np.maximum(1,2*tp+fp+fn),'false_positive':fp,'false_negative':fn,
                            'threshold':thresholds})
    per_label.to_csv(folder/'per_label.csv',index=False)
    errors=[]
    for j,i in enumerate(CAL_IDX):
        missed=[LABELS[k] for k in np.flatnonzero(y[j]&~pred[j])]
        extra=[LABELS[k] for k in np.flatnonzero(~y[j]&pred[j])]
        if missed or extra: errors.append({'id':records['dev'][i]['id'],'missed':missed,'extra':extra})
    atomic_json(errors,folder/'errors.json')
    row={k:v for k,v in summary.items() if not isinstance(v,dict)}
    row.update({'calibrated_'+k:v for k,v in summary['calibration_per_label'].items()})
    row['mean_predicted_labels']=float(pred.sum(1).mean())
    row['mean_gold_labels']=float(y.sum(1).mean())
    comparison.append(row)
table=pd.DataFrame(comparison).sort_values('best_selection_macro_f1',ascending=False)
table.to_csv(WORK/'experiment_comparison.csv',index=False); display(table)
cooccurrence=Y_TRAIN.T@Y_TRAIN
pd.DataFrame(cooccurrence.astype(int),index=LABELS,columns=LABELS).to_csv(WORK/'label_cooccurrence.csv')
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
table.plot.bar(x='method',y=['best_selection_macro_f1','calibrated_macro_f1'],ax=axes[0],rot=30)
axes[0].set_ylim(0,1); axes[0].set_title('Selection @0.5 vs threshold-tuned calibration')
table.plot.bar(x='method',y='trainable_parameters',ax=axes[1],rot=30,legend=False)
axes[1].set_title('Trainable parameters')
fig.tight_layout(); fig.savefig(WORK/'comparison.png',dpi=160); plt.show()


## 11. Exact submission format and inference artifacts
By default only the selected winner is exported; set EXPORT_ALL_VARIANTS=True for all six.
Every test ID appears exactly once, as `{"id": "...", "pred": [integer statute IDs]}`.
ZIP contains only `prediction.json` at its root. In smoke mode inference is limited and named
`SMOKE_NOT_FOR_SUBMISSION.zip`; real mode always predicts every official test document.
Each run retains best weights, full recovery checkpoint, vocabulary, thresholds, provenance,
timing and metrics. Keep this self-contained notebook, the original Phase 1 archive and run
directory together to reconstruct inference. `load_trained(run_dir)` is the reload entry point.
Optional official evaluation: download the course evaluator and run its documented CLI against
a dev prediction and reference file; its CLI is not guessed or silently executed here.


In [ ]:
def validate_submission(predictions, expected_ids):
    assert isinstance(predictions,list)
    ids=[r['id'] for r in predictions]
    assert len(ids)==len(set(ids))==len(expected_ids) and set(ids)==set(expected_ids)
    valid=set(LABELS)
    for r in predictions:
        assert set(r)=={'id','pred'} and isinstance(r['pred'],list)
        assert all(type(v) is int and v in valid for v in r['pred'])
        assert len(r['pred'])==len(set(r['pred']))

best_method=max(RESULTS,key=lambda k:RESULTS[k][1]['best_selection_macro_f1'])
EXPORT_METHODS = list(RESULTS) if EXPORT_ALL_VARIANTS else [best_method]
for method in EXPORT_METHODS:
    folder,summary=RESULTS[method]
    model=load_trained(folder)
    ts=json.loads((folder/'thresholds.json').read_text()); assert ts['label_ids']==LABELS
    thresholds=np.array(ts['thresholds'])
    indices=np.arange(min(3,len(records['test']))) if CFG['smoke_only'] else np.arange(len(records['test']))
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    began=time.perf_counter(); logits=predict_logits(model,'test',indices)
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    seconds=time.perf_counter()-began
    predictions=[{'id':records['test'][int(i)]['id'],'pred':[LABELS[j] for j in np.flatnonzero(p>=thresholds)]}
                 for i,p in zip(indices,sigmoid(logits))]
    validate_submission(predictions,[records['test'][int(i)]['id'] for i in indices])
    atomic_json(predictions,folder/'prediction.json')
    archive=folder/('SMOKE_NOT_FOR_SUBMISSION.zip' if CFG['smoke_only'] else 'submission.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        z.write(folder/'prediction.json','prediction.json')
    with zipfile.ZipFile(archive) as z:
        assert z.namelist()==['prediction.json']
        validate_submission(json.loads(z.read('prediction.json')),[records['test'][int(i)]['id'] for i in indices])
    atomic_json({'documents':len(indices),'seconds':seconds,'documents_per_second':len(indices)/max(seconds,1e-9)},folder/'inference_timing.json')
    atomic_json(LABELS,folder/'labels.json')
    # Labeled dev predictions for comparison with the official evaluator.
    dev_logits=predict_logits(model,'dev',SELECT_IDX)
    atomic_json([{'id':records['dev'][int(i)]['id'],'pred':[LABELS[j] for j in np.flatnonzero(p>=thresholds)]}
                 for i,p in zip(SELECT_IDX,sigmoid(dev_logits))],folder/'dev_prediction.json')
    atomic_json([{'id':records['dev'][int(i)]['id'],'labels':records['dev'][int(i)]['labels']} for i in SELECT_IDX],folder/'dev_reference.json')
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    backup(folder); display(FileLink(str(archive)))
best_method=max(RESULTS,key=lambda k:RESULTS[k][1]['best_selection_macro_f1'])
atomic_json({'recommended_method_by_selection':best_method,'experiments':list(RESULTS),
             'all_four_architectures':False,'scope_note':'LEXWeave-only by user choice; architecture requirement unmet',
             'five_adaptation_methods':set(['full','partial','lorma','adapter','prompt']).issubset(RESULTS),
             'knowledge_distillation':'kd' in RESULTS,'smoke_only':CFG['smoke_only']},WORK/'completion_audit.json')
atomic_json({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,
             'numpy':np.__version__,'tokenizers':__import__('tokenizers').__version__,
             'device':torch.cuda.get_device_name() if DEVICE.type=='cuda' else 'cpu'},WORK/'environment.json')
print('Recommended by selection-half macro-F1:',best_method)
print('Outputs:',WORK,'| Architecture coverage: LEXWeave only (requirement unmet).')
